KRAMA Framework

In [ ]:
#Installing needed libraries and dependencies
!rm -rf IndicTransToolkit
!pip uninstall -y indictranstoolkit
!pip install indictranstoolkit
!pip install "vllm==0.6.4.post1" "transformers==4.46.3" "tokenizers==0.20.3" accelerate pandas datasets scipy sentence-transformers faiss-cpu

In [ ]:
import os
import gc
import json
import pickle
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
from vllm import LLM, SamplingParams
from IndicTransToolkit import IndicProcessor

from google.colab import drive
drive.mount('/content/drive')

WORKSPACE_DIR = "PATH_FOR_WORKSPACE_DIRECTORY_WHERE_EVERYTHING_WILL_BE_STORED"
RAG_DIR = "PATH_FOR_WHERE_MEGA_RAG_IS_STORED"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

#JSON-Straitjacket
def load_straitjacket(stage_name):
    if stage_name == "stage_3_scorer":
        stage_name = "stage_3_scorer_Llama1B"
    path = os.path.join(WORKSPACE_DIR, f"{stage_name}.json")

    if not os.path.exists(path):
        print(f"Missing input state: {path}. Triggering generation...")
        if stage_name == "stage_0_init":
            run_stage_0_init()
        elif stage_name == "stage_1_translated":
            run_stage_1_translation()
        elif stage_name == "stage_2_teacher":
            run_stage_2_teacher()
        elif stage_name == "stage_3_scorer_Llama1B":
            run_stage_3_scorer()
        else:
            raise FileNotFoundError(f"Missing input state and no generation mapping for: {stage_name}")

    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_straitjacket(data, stage_name):
    path = os.path.join(WORKSPACE_DIR, f"{stage_name}.json")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Straitjacket locked for {stage_name} at {path}")

#Dataset
def run_stage_0_init():
    print("Executing Stage 0: Initialization...")
    dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Validation_Or_Test_Split", token=True)
    df_pipeline = dataset.to_pandas()
    df_pipeline = df_pipeline.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Total samples loaded: {len(df_pipeline)}")
    save_straitjacket(df_pipeline.to_dict(orient='records'), "stage_0_init")

#Tier 1
def run_stage_1_translation():
    print("Executing Stage 1: Fast Translation...")
    data = load_straitjacket("stage_0_init")

    from IndicTransToolkit import IndicProcessor

    MODEL_NAME = "ai4bharat/indictrans2-indic-en-1B" #Translation Model
    SRC_LANG, TGT_LANG = "tel_Telu", "eng_Latn"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    ip = IndicProcessor(inference=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float16
    ).to(DEVICE)

    texts = [row['question'] for row in data]
    text_indices = sorted(range(len(texts)), key=lambda k: len(texts[k]))
    sorted_texts = [texts[i] for i in text_indices]

    translated_texts = []
    batch_size = 32

    for i in range(0, len(sorted_texts), batch_size):
        batch = sorted_texts[i : i + batch_size]
        processed_batch = ip.preprocess_batch(batch, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

        inputs = tokenizer(
            processed_batch, truncation=True, padding="longest",
            return_tensors="pt", return_attention_mask=True,
        ).to(DEVICE)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs, use_cache=False, min_length=0, max_length=256,
                num_beams=1, do_sample=False, num_return_sequences=1,
            )

        decoded = tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        post_processed = ip.postprocess_batch(decoded, lang=TGT_LANG)
        translated_texts.extend(post_processed)

    final_results = [None] * len(texts)
    for i, original_idx in enumerate(text_indices):
        final_results[original_idx] = translated_texts[i]

    for i, row in enumerate(data):
        row['question_en'] = final_results[i]

    save_straitjacket(data, "stage_1_translated")
    del model, tokenizer, ip
    flush_memory()

#Tier 2
def run_stage_2_teacher():
    print("Executing Stage 2: vLLM Rationale Generation...")
    data = load_straitjacket("stage_1_translated")

    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ" #Rationale Generation Model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    llm = LLM(
        model=MODEL_NAME, quantization="awq", dtype="half",
        gpu_memory_utilization=0.90, max_model_len=4096, max_num_seqs=8,
        trust_remote_code=True, enforce_eager=True, disable_custom_all_reduce=True
    )
    #Prompt
    prompts = []
    for row in data:
        messages = [
            {"role": "system", "content": "You are a professional educational and guidance expert. Provide a concise, formal rationale explaining the concepts behind the user's question. Do NOT use inner monologue. Do NOT state the final correct answer or mention any options. Provide only the objective explanation."},
            {"role": "user", "content": f"Question: {row['question_en']}"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(text)

    sampling_configs = {
        "logical": SamplingParams(temperature=0.2, top_p=0.95, max_tokens=300),
        "balanced": SamplingParams(temperature=0.45, top_p=0.95, max_tokens=350),
        "creative": SamplingParams(temperature=0.7, top_p=0.90, max_tokens=400)
    }

    for mode, params in sampling_configs.items():
        print(f"Generating {mode} rationales...")
        outputs = llm.generate(prompts, params, use_tqdm=True)
        for i, output in enumerate(outputs):
            data[i][f'rationale_{mode}'] = output.outputs[0].text.strip()

        save_straitjacket(data, f"stage_2_teacher_checkpoint_{mode}")

    save_straitjacket(data, "stage_2_teacher")
    del llm
    flush_memory()

#Tier 3

### Copy paste the Tier 3 code from bottom of the notebook, depending upon the particular model you want to use.

#Tier 4
def run_stage_4_critic():
    print("Executing Stage 4: Critic & RAG Integration (vLLM + Probability Gate)...")
    import math
    data = load_straitjacket("stage_3_scorer")

    #Retrieval
    index_path = os.path.join(RAG_DIR, "mega_corpus_m3.index")
    corpus_path = os.path.join(RAG_DIR, "mega_metadata.pkl")
    print("Loading FAISS Index and Corpus...")
    index = faiss.read_index(index_path)
    with open(corpus_path, 'rb') as f:
        corpus_data = pickle.load(f)
    print("Loading Retriever (BGE-M3)...")
    retriever = SentenceTransformer("BAAI/bge-m3").to("cuda")

    prompts = []

    for i, row in enumerate(data):
        q_emb = retriever.encode([row['question_en']], normalize_embeddings=True)
        similarities, doc_indices = index.search(q_emb, k=1)
        top_score = similarities[0][0]
        top_doc = corpus_data[doc_indices[0][0]]

        row['rag_score'] = float(top_score)

        prompt_start = f"""You are a strict and knowledgable evaluator. Analyze the given information and provide the final correct option letter (A, B, C, or D).
English Question: {row['question_en']}
"""
        if top_score >= 0.85:
            prompt_context = f"Knowledge Context: {top_doc}\n"
        else:
            prompt_context = ""

        prompt_end = f"""Proposed Logical Rationale: {row['rationale_logical']}
Options:
A) {row['option1']}
B) {row['option2']}
C) {row['option3']}
D) {row['option4']}
Current Ensemble Prediction: {row['ensemble_pred']}

Refined Final Answer (Letter only):"""

        critic_prompt = prompt_start + prompt_context + prompt_end
        prompts.append(critic_prompt)
    print("Retrieval complete. Purging BGE-M3 from memory...")
    del retriever, index, corpus_data
    flush_memory()
    print("Loading Sarvam Critic via vLLM...")
    critic_id = "sarvamai/sarvam-1"

    llm = LLM(
        model=critic_id,
        dtype="half",
        gpu_memory_utilization=0.85,
        max_model_len=4096,
        enforce_eager=True,
        disable_custom_all_reduce=True,
        trust_remote_code=True
    )

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=1,
        logprobs=5
    )

    #Critic
    print("Batch generating Critic answers with logprobs...")
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    critic_overrides = 0
    confidence_fallbacks = 0

    for i, output in enumerate(outputs):
        ensemble_pred = data[i]['ensemble_pred']

        raw_text = output.outputs[0].text.strip().upper()
        valid_options = ["A", "B", "C", "D"]
        parsed_critic_result = next((char for char in raw_text if char in valid_options), None)

        try:
            first_token_logprobs = output.outputs[0].logprobs[0]
            gen_token_id = output.outputs[0].token_ids[0]
            logprob_val = first_token_logprobs[gen_token_id].logprob
            confidence_score = math.exp(logprob_val)
        except (IndexError, KeyError, TypeError):
            confidence_score = 0.0

        if parsed_critic_result and confidence_score >= 0.60:
            data[i]['final_answer'] = parsed_critic_result
            critic_overrides += 1
        else:
            data[i]['final_answer'] = ensemble_pred
            confidence_fallbacks += 1

        data[i]['critic_confidence'] = float(confidence_score)

    print(f"\nCritic confidently answered (>=0.60): {critic_overrides} times.")
    print(f"Critic rejected due to low confidence (<0.60): {confidence_fallbacks} times. (Fell back to Ensemble)")

    save_straitjacket(data, "stage_4_critic_final")

    final_df = pd.DataFrame(data)
    final_csv_path = os.path.join(WORKSPACE_DIR, "pipeline_final_output_Llama1B.csv")
    final_df.to_csv(final_csv_path, index=False)
    print(f"Pipeline completely executed. Output saved to {final_csv_path}")

    del llm
    flush_memory()

#Run Framework
if __name__ == "__main__":
    run_stage_4_critic()

In [ ]:
#Tier-wise Accuracy Calculation
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

csv_path = "PATH_WHERE_FINAL_RESULT_IS_STORED"
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: Could not find the file at {csv_path}. Ensure the pipeline completed.")
    exit()

def resolve_target_label(target_val):
    target_str = str(target_val).strip().lower()
    mapping = {
        'option1': 'A',
        'option2': 'B',
        'option3': 'C',
        'option4': 'D'
    }
    return mapping.get(target_str, "UNKNOWN")

df['target_clean'] = df['target'].apply(resolve_target_label)

def extract_letter(pred):
    pred = str(pred).strip().upper()
    for char in pred:
        if char in ['A', 'B', 'C', 'D']:
            return char
    return "UNKNOWN"

df['ensemble_pred_clean'] = df['ensemble_pred'].apply(extract_letter)
df['final_answer_clean'] = df['final_answer'].apply(extract_letter)
total_samples = len(df)
valid_targets = len(df[df['target_clean'] != "UNKNOWN"])

if valid_targets < total_samples:
    print(f"WARNING: {total_samples - valid_targets} targets could not be mapped. Check dataset formatting.")
final_correct = (df['final_answer_clean'] == df['target_clean']).sum()
ensemble_correct = (df['ensemble_pred_clean'] == df['target_clean']).sum()

final_acc = (final_correct / total_samples) * 100
ensemble_acc = (ensemble_correct / total_samples) * 100
critic_delta = final_acc - ensemble_acc

print("=== OVERALL ABLATION METRICS ===")
print(f"Total Samples Processed: {total_samples}")
print(f"Pre-Critic (Ensemble Only) Accuracy: {ensemble_acc:.2f}%")
print(f"Full Pipeline (Ensemble + Critic + RAG) Accuracy: {final_acc:.2f}%")
print(f"Critic/RAG Contribution (Delta): {critic_delta:+.2f}%\n")

print("=== DOMAIN-WISE ACCURACY (FULL PIPELINE) ===")
domain_stats = []
if 'domain' in df.columns:
    for domain in df['domain'].unique():
        domain_df = df[df['domain'] == domain]
        d_total = len(domain_df)
        if d_total > 0:
            d_correct = (domain_df['final_answer_clean'] == domain_df['target_clean']).sum()
            d_acc = (d_correct / d_total) * 100
            domain_stats.append({'Domain': domain, 'Samples': d_total, 'Accuracy': d_acc})

    domain_results_df = pd.DataFrame(domain_stats).sort_values(by='Accuracy', ascending=False)
    print(domain_results_df.to_string(index=False))
else:
    print("No 'domain' column found in dataset.")
critic_failures = df[(df['ensemble_pred_clean'] == df['target_clean']) &
                     (df['final_answer_clean'] != df['target_clean'])]

print(f"\n=== CRITIC REGRESSIONS ===")
print(f"Number of times the Critic overturned a correct prediction: {len(critic_failures)}")
if len(critic_failures) > 0:
    print("Sample regression rows (Indices):", critic_failures.index.tolist()[:5])

TIER 3 CODES FOR ALL MODELS

Gemma-2-2B-it OR Sarvam-1

In [ ]:
def run_stage_3_scorer():
    print("Executing Stage 3: Scorer Decision Ensemble...")
    data = load_straitjacket("stage_2_teacher")

    scorer_id = "google/gemma-2-2b-it" #OR sarvamai/sarvam-1
    tokenizer = AutoTokenizer.from_pretrained(scorer_id)
    model = AutoModelForCausalLM.from_pretrained(scorer_id, torch_dtype=torch.float16, device_map="cuda")

    token_ids = {
        "A": tokenizer.encode("A", add_special_tokens=False)[-1],
        "B": tokenizer.encode("B", add_special_tokens=False)[-1],
        "C": tokenizer.encode("C", add_special_tokens=False)[-1],
        "D": tokenizer.encode("D", add_special_tokens=False)[-1]
    }

    def get_vote(question, rationale, options):
        content = f"Question: {question}\nReasoning: {rationale}\nOptions:\nA) {options[0]}\nB) {options[1]}\nC) {options[2]}\nD) {options[3]}\nYou are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D):"
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :]
            scores = {letter: logits[t_id].item() for letter, t_id in token_ids.items()}
            return max(scores, key=scores.get)

    for i, row in enumerate(data):
        opts = [row['option1'], row['option2'], row['option3'], row['option4']]
        votes = [
            get_vote(row['question_en'], row['rationale_logical'], opts),
            get_vote(row['question_en'], row['rationale_balanced'], opts),
            get_vote(row['question_en'], row['rationale_creative'], opts)
        ]

        vote_counts = {v: votes.count(v) for v in set(votes)}
        max_count = max(vote_counts.values())
        leaders = [v for v, count in vote_counts.items() if count == max_count]
        row['ensemble_pred'] = votes[0] if votes[0] in leaders else leaders[0]

    save_straitjacket(data, "stage_3_scorer_Gemma2B") #Change file name as per model
    del model, tokenizer
    flush_memory()

Llama-3.1-1B-Instruct

In [ ]:
def run_stage_3_scorer():
    print("Executing Stage 3: Scorer Decision Ensemble...")
    data = load_straitjacket("stage_2_teacher")

    scorer_id = "meta-llama/Llama-3.2-1B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(scorer_id)
    model = AutoModelForCausalLM.from_pretrained(scorer_id, torch_dtype=torch.float16, device_map="cuda")

    token_ids = {
        "A": tokenizer.encode("A", add_special_tokens=False)[0],
        "B": tokenizer.encode("B", add_special_tokens=False)[0],
        "C": tokenizer.encode("C", add_special_tokens=False)[0],
        "D": tokenizer.encode("D", add_special_tokens=False)[0]
    }

    def get_vote(question, rationale, options):
        content = f"Question: {question}\nReasoning: {rationale}\nOptions:\nA) {options[0]}\nB) {options[1]}\nC) {options[2]}\nD) {options[3]}\nYou are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D):"
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :]
            scores = {letter: logits[t_id].item() for letter, t_id in token_ids.items()}
            return max(scores, key=scores.get)

    for i, row in enumerate(data):
        opts = [row['option1'], row['option2'], row['option3'], row['option4']]
        votes = [
            get_vote(row['question_en'], row['rationale_logical'], opts),
            get_vote(row['question_en'], row['rationale_balanced'], opts),
            get_vote(row['question_en'], row['rationale_creative'], opts)
        ]

        vote_counts = {v: votes.count(v) for v in set(votes)}
        max_count = max(vote_counts.values())
        leaders = [v for v, count in vote_counts.items() if count == max_count]
        row['ensemble_pred'] = votes[0] if votes[0] in leaders else leaders[0]

    save_straitjacket(data, "stage_3_scorer_Llama1B")
    del model, tokenizer
    flush_memory()

Llama-3.1-8B-Instruct OR Aya-23-8B (AWQ Versions) OR Llama-3.2-3B-Instruct

In [ ]:
def run_stage_3_scorer():
    print("Executing Stage 3: Scorer Decision Ensemble (vLLM Batched)...")
    data = load_straitjacket("stage_2_teacher")

    scorer_id = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4" #OR alijawad07/aya-23-8B-AWQ-GEMM
    tokenizer = AutoTokenizer.from_pretrained(scorer_id)

    llm = LLM(
        model=scorer_id,
        quantization="awq", #Remove if Llama-3.2-3B-Instruct
        dtype="half",
        gpu_memory_utilization=0.90, #You can configure as needed
        max_model_len=4096,
        enforce_eager=True,
        disable_custom_all_reduce=True
    )

    token_ids = {
        "A": tokenizer.encode("A", add_special_tokens=False)[-1],
        "B": tokenizer.encode("B", add_special_tokens=False)[-1],
        "C": tokenizer.encode("C", add_special_tokens=False)[-1],
        "D": tokenizer.encode("D", add_special_tokens=False)[-1]
    }

    all_prompts = []
    for row in data:
        opts = [row['option1'], row['option2'], row['option3'], row['option4']]
        for mode in ['rationale_logical', 'rationale_balanced', 'rationale_creative']:
            content = f"Question: {row['question_en']}\nReasoning: {row[mode]}\nOptions:\nA) {opts[0]}\nB) {opts[1]}\nC) {opts[2]}\nD) {opts[3]}\nYou are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D):"
            prompt = tokenizer.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
            all_prompts.append(prompt)

    sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=5)
    print("Batch generating ensemble votes...")
    outputs = llm.generate(all_prompts, sampling_params, use_tqdm=True)

    out_idx = 0
    for row in data:
        votes = []
        for _ in range(3):
            output = outputs[out_idx]
            out_idx += 1

            scores = {letter: -999.0 for letter in ["A", "B", "C", "D"]}
            if output.outputs[0].logprobs:
                first_token_logprobs = output.outputs[0].logprobs[0]
                for letter, t_id in token_ids.items():
                    if t_id in first_token_logprobs:
                        scores[letter] = first_token_logprobs[t_id].logprob

            best_vote = max(scores, key=scores.get)

            if all(v == -999.0 for v in scores.values()):
                gen_text = output.outputs[0].text.strip().upper()
                best_vote = gen_text if gen_text in ["A", "B", "C", "D"] else "A"

            votes.append(best_vote)

        vote_counts = {v: votes.count(v) for v in set(votes)}
        max_count = max(vote_counts.values())
        leaders = [v for v, count in vote_counts.items() if count == max_count]
        row['ensemble_pred'] = votes[0] if votes[0] in leaders else leaders[0]

    save_straitjacket(data, f"stage_3_scorer_Llama8B") #Change file name as per model
    del llm
    flush_memory()